In [ ]:
# Exact versions, tested together on Python 3.14 in the course .venv.
# One line on purpose: line magics do not take backslash continuations.
%pip install -q langgraph==1.2.11 langchain-core==1.6.3 langchain-openai==1.6.2 langchain-text-splitters==1.1.2 chromadb==1.5.9 onnxruntime==1.29.0 pymupdf==1.27.2.3 python-dotenv==1.2.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 113.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9

In [ ]:
import os
import textwrap
from pathlib import Path
from typing import TypedDict

import pymupdf
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import StateGraph, START, END
from google.colab import userdata


load_dotenv()

# --- knobs ------------------------------------------------------------------
PDF_PATH = Path("/content/company_handbook.pdf")
CHROMA_DIR = "chroma_selfrag"
COLLECTION = "handbook"
K = 4
MAX_ATTEMPTS = 2
MODEL = os.environ.get("MODEL_NAME", "openai/gpt-oss-120b")
GROQ_URL = "https://api.groq.com/openai/v1"                        # Groq endpoint

assert PDF_PATH.exists(), f"No PDF at {PDF_PATH}. Put one there or change PDF_PATH."

# Ensure GROQ_API_KEY (actually holding the Groq key) is available in os.environ
if not os.environ.get("GROQ_API_KEY"):
    colab_key = userdata.get('GROQ_API_KEY')
    if colab_key:
        os.environ["GROQ_API_KEY"] = colab_key
    else:
        raise AssertionError("GROQ_API_KEY not found in environment or Colab secrets.")

assert os.environ.get("GROQ_API_KEY"), "GROQ_API_KEY is still not set after attempting to load from Colab secrets."

def llm():
    """One shared chat model. Temperature 0 keeps the routing decisions stable."""
    return ChatOpenAI(
        model=MODEL,
        base_url=GROQ_URL,
        api_key=os.environ["GROQ_API_KEY"],   # holds the Groq key, name kept as-is
        temperature=0,
    )


def ask(prompt: str) -> str:
    """Send one prompt to the model and return the reply as plain text."""
    return llm().invoke([HumanMessage(content=prompt)]).content.strip()


print("model:", MODEL)
print("pdf:  ", PDF_PATH)

model: openai/gpt-oss-120b
pdf:   /content/company_handbook.pdf


In [ ]:
doc = pymupdf.open(PDF_PATH)
pages = [{"page": i + 1, "text": page.get_text()} for i, page in enumerate(doc)]
doc.close()

print(f"{len(pages)} pages, {sum(len(p['text']) for p in pages):,} characters")
print("-" * 70)
print(pages[1]["text"][:500])

4 pages, 2,866 characters
----------------------------------------------------------------------
Section 4. Annual Leave
Employees accrue 20 days of paid annual leave per calendar year, plus public holidays.
Leave accrues monthly and is available from the first month of employment.
Section 4.1 Carryover. Employees may carry over up to 5 unused annual leave days into the
next calendar year. Carried over days must be used by March 31 of the following year. After
March 31, any remaining carried over days expire and are removed from the balance.
Section 4.2 Requesting leave. Leave requests are 


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = []
for page in pages:
    for j, piece in enumerate(splitter.split_text(page["text"])):
        chunks.append({"id": f"p{page['page']}-c{j}", "text": piece, "page": page["page"]})

print(f"{len(chunks)} chunks")
print("-" * 70)
print(chunks[1]["id"], "|", chunks[1]["text"][:220], "...")

5 chunks
----------------------------------------------------------------------
p2-c0 | Section 4. Annual Leave
Employees accrue 20 days of paid annual leave per calendar year, plus public holidays.
Leave accrues monthly and is available from the first month of employment.
Section 4.1 Carryover. Employees m ...


In [ ]:
client = chromadb.PersistentClient(path=CHROMA_DIR)

# Chroma ships a small ONNX embedding model (all-MiniLM-L6-v2). No torch needed.
# It downloads about 80 MB the first time and caches it under ~/.cache/chroma.
embedder = embedding_functions.DefaultEmbeddingFunction()

try:
    client.delete_collection(COLLECTION)
except Exception:
    pass

collection = client.create_collection(name=COLLECTION, embedding_function=embedder)
collection.add(
    ids=[c["id"] for c in chunks],
    documents=[c["text"] for c in chunks],
    metadatas=[{"page": c["page"]} for c in chunks],
)

print("vectors in the store:", collection.count())

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:06<00:00, 12.5MiB/s]


vectors in the store: 5


In [ ]:
def retrieve(query: str, k: int = K) -> list[dict]:
    """The agent's only tool. Returns the k nearest chunks for a query."""
    res = collection.query(query_texts=[query], n_results=k)
    hits = []
    for text, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        hits.append({"text": text, "page": meta["page"], "score": round(1 - dist, 3)})
    return hits


for h in retrieve("carry over unused vacation days"):
    print(f"p{h['page']}  score={h['score']}  {h['text'][:80]}...")

p2  score=0.144  Section 4. Annual Leave
Employees accrue 20 days of paid annual leave per calend...
p2  score=0.049  approval.
Section 4.3 Policy year 2022. Under the 2022 rules, employees could ca...
p3  score=-0.018  Section 5. Policy Updates
Section 5.1 Carryover amendment, 2023. Starting with t...
p4  score=-0.457  Section 6. Expenses, Refunds and Support
Section 6.1 Travel refunds. Approved tr...


In [ ]:
class AgentState(TypedDict):
    question: str
    needs_retrieval: bool
    source:str
    strategy: str
    search_queries: list[str]
    chunks: list[dict]
    grade: str
    attempts: int
    answer: str
    trace: list[str]

In [ ]:
import requests

SERPER_API_KEY = os.environ.get("SERPER_API_KEY")
if not SERPER_API_KEY:
    colab_key = userdata.get("SERPER_API_KEY")
    if colab_key:
        os.environ["SERPER_API_KEY"] = colab_key
        SERPER_API_KEY = colab_key
    else:
        raise AssertionError("SERPER_API_KEY not found in environment or Colab secrets.")


def web_search(query: str, k: int = 4) -> list[dict]:
    """Search the live web via serper.dev. Same shape as retrieve()."""
    resp = requests.post(
        "https://google.serper.dev/search",
        headers={"X-API-KEY": SERPER_API_KEY, "Content-Type": "application/json"},
        json={"q": query},
        timeout=15,
    )
    resp.raise_for_status()
    data = resp.json()
    hits = []
    for item in data.get("organic", [])[:k]:
        hits.append({
            "text": item.get("snippet", ""),
            "page": item.get("link", ""),   # reused field, holds the URL here
            "score": None,
        })
    return hits

In [ ]:
def decide_retrieval(state: AgentState) -> str:
    """Return "direct", "kb", or "web"."""
    prompt = (
        "You have two tools:\n"
        "1) a knowledge base: the Northwind employee handbook (company policy, "
        "leave, refunds, tickets, staff)\n"
        "2) a live web search tool for anything current or outside the company\n\n"
        f"Question: {state['question']}\n\n"
        "Decide:\n"
        "- DIRECT: you can answer confidently from general knowledge already "
        "(greetings, simple math, well-known facts)\n"
        "- KB: the question is about Northwind company policy\n"
        "- WEB: the question needs current or external info not in the handbook\n\n"
        "Answer with exactly one word: DIRECT, KB, or WEB."
    )
    reply = ask(prompt).strip().upper()
    if reply.startswith("KB"):
        return "kb"
    if reply.startswith("WEB"):
        return "web"
    return "direct"
def choose_strategy(state: AgentState) -> str:
    """Pick the search strategy for this question."""
    prompt = (
        "Pick ONE search strategy for the question below.\n"
        "rewrite = clean keyword query\n"
        "hyde = write a fake answer and search with it\n"
        "decompose = split a multi-part question into sub-queries\n"
        f"Question: {state['question']}\n"
        "Answer with one word."
    )
    if state.get("attempts", 0) > 0:
        prompt += f"\nYou already tried {state['strategy']}. Pick a different one."

    reply = ask(prompt).strip().lower()
    for option in ("rewrite", "hyde", "decompose"):
        if option in reply:
            return option
    return "rewrite"  # safe fallback
def rewrite_query(state: AgentState) -> str:
    """Return one cleaned up search query for the question."""
    prompt = (
        "Rewrite the question below as a short search query made of keywords, "
        "the way you would type it into a search box. "
        "Return the query only, no quotes, no explanation.\n"
        f"Question: {state['question']}"
    )
    reply = ask(prompt).strip()
    return reply.strip('"').strip("'").strip()
def make_hyde(state: AgentState) -> str:
    """Write a short hypothetical answer and return it as the search query."""
    prompt = (
        "Write the first two or three sentences of the handbook section that "
        "would answer this question. It does not need to be correct, it just "
        "needs to sound like the document.\n"
        f"Question: {state['question']}"
    )
    return ask(prompt).strip()
def decompose(state: AgentState) -> list[str]:
    """Split the question into 2 to 4 standalone sub-queries."""
    prompt = (
        "Split the question below into 2 to 4 standalone search sub-queries, "
        "one per line, no numbering, no extra text.\n"
        f"Question: {state['question']}"
    )
    reply = ask(prompt)
    queries = [
        line.strip("-* 0123456789.").strip()
        for line in reply.splitlines()
        if line.strip()
    ]
    if len(queries) < 2:
        return [state["question"]]
    return queries[:4]
def grade_chunks(state: AgentState) -> str:
    """Return "good" if the chunks can answer the question, else "bad"."""
    context = "\n\n".join(f"[p{c['page']}] {c['text']}" for c in state["chunks"])
    prompt = (
        f"Question: {state['question']}\n\n"
        f"Passages:\n{context}\n\n"
        "Do these passages contain the specific facts (numbers, dates, names) "
        "needed to answer the question? Be strict — if a key fact is missing, "
        "say BAD. Answer GOOD or BAD only."
    )
    reply = ask(prompt).strip().upper()
    return "good" if reply.startswith("GOOD") else "bad"
def generate_answer(state: AgentState) -> str:
    if not state["chunks"]:
        prompt = f"Answer this question in one short sentence, from general knowledge:\n{state['question']}"
        return ask(prompt).strip()

    if state["source"] == "web":
        context = "\n\n".join(f"[{c['page']}] {c['text']}" for c in state["chunks"])
        cite = "Cite the source URL in brackets like [https://example.com] for each fact."
    else:
        context = "\n\n".join(f"[p{c['page']}] {c['text']}" for c in state["chunks"])
        cite = "Cite the page for every fact like [p2]."

    status = (
        "These passages do NOT contain enough information. Say clearly what is missing."
        if state.get("grade") == "bad"
        else f"Answer the question using only the passages above. {cite} If something is not covered, say so instead of guessing."
    )

    prompt = f"Question: {state['question']}\n\nPassages:\n{context}\n\n{status}"
    return ask(prompt).strip()


In [ ]:
def n_decide(state: AgentState) -> AgentState:
    source = decide_retrieval(state)
    trace = state["trace"] + [f"decide_retrieval -> {source}"]
    return {
        **state,
        "source": source,
        "needs_retrieval": source != "direct",
        "trace": trace,
    }


def n_strategy(state: AgentState) -> AgentState:
    strategy = choose_strategy(state)
    trace = state["trace"] + [f"choose_strategy -> {strategy}"]
    return {**state, "strategy": strategy, "trace": trace}


def n_rewrite(state: AgentState) -> AgentState:
    query = rewrite_query(state)
    hits = retrieve(query)
    trace = state["trace"] + [f"rewrite_query -> {query!r} ({len(hits)} hits)"]
    return {
        **state,
        "search_queries": [query],
        "chunks": hits,
        "attempts": state["attempts"] + 1,
        "trace": trace,
    }


def n_hyde(state: AgentState) -> AgentState:
    query = make_hyde(state)
    hits = retrieve(query)
    trace = state["trace"] + [f"make_hyde -> {query!r} ({len(hits)} hits)"]
    return {
        **state,
        "search_queries": [query],
        "chunks": hits,
        "attempts": state["attempts"] + 1,
        "trace": trace,
    }


def n_decompose(state: AgentState) -> AgentState:
    queries = decompose(state)
    hits = []
    for q in queries:
        hits.extend(retrieve(q))
    trace = state["trace"] + [f"decompose -> {queries} ({len(hits)} hits total)"]
    return {
        **state,
        "search_queries": queries,
        "chunks": hits,
        "attempts": state["attempts"] + 1,
        "trace": trace,
    }


def n_grade(state: AgentState) -> AgentState:
    grade = grade_chunks(state)
    trace = state["trace"] + [f"grade_chunks -> {grade}"]
    return {**state, "grade": grade, "trace": trace}


def n_answer(state: AgentState) -> AgentState:
    answer = generate_answer(state)
    trace = state["trace"] + ["generate_answer -> done"]
    return {**state, "answer": answer, "trace": trace}
def n_web(state: AgentState) -> AgentState:
    query = rewrite_query(state)   # reuse the same rewrite for a clean query
    hits = web_search(query)
    trace = state["trace"] + [f"web_search -> {query!r} ({len(hits)} hits)"]
    return {
        **state,
        "search_queries": [query],
        "chunks": hits,
        "attempts": state["attempts"] + 1,
        "trace": trace,
    }
def route_after_decide(state: AgentState) -> str:
    if state["source"] == "kb":
        return "strategy"
    if state["source"] == "web":
        return "web"
    return "answer"
def route_after_grade(state: AgentState) -> str:
    if state["grade"] == "good" or state["attempts"] >= MAX_ATTEMPTS:
        return "answer"
    return "strategy" if state["source"] == "kb" else "web"
def route_strategy(state: AgentState) -> str:
    return state["strategy"]
graph = StateGraph(AgentState)

graph.add_node("decide", n_decide)
graph.add_node("strategy", n_strategy)
graph.add_node("rewrite", n_rewrite)
graph.add_node("hyde", n_hyde)
graph.add_node("decompose", n_decompose)
graph.add_node("web", n_web)          # NEW
graph.add_node("grade", n_grade)
graph.add_node("answer", n_answer)

graph.add_edge(START, "decide")

graph.add_conditional_edges(
    "decide", route_after_decide,
    {"strategy": "strategy", "web": "web", "answer": "answer"},   # updated
)

graph.add_conditional_edges(
    "strategy", route_strategy,
    {"rewrite": "rewrite", "hyde": "hyde", "decompose": "decompose"},
)

graph.add_edge("rewrite", "grade")
graph.add_edge("hyde", "grade")
graph.add_edge("decompose", "grade")
graph.add_edge("web", "grade")        # NEW

graph.add_conditional_edges(
    "grade", route_after_grade,
    {"answer": "answer", "strategy": "strategy", "web": "web"},   # updated
)

graph.add_edge("answer", END)

app = graph.compile()
def run_agent(question: str) -> AgentState:
    initial_state: AgentState = {
        "question": question,
        "needs_retrieval": False,
        "source": "",
        "strategy": "",
        "search_queries": [],
        "chunks": [],
        "grade": "",
        "attempts": 0,
        "answer": "",
        "trace": [],
    }
    return app.invoke(initial_state)

In [ ]:
TEST_PROMPTS = [
    "hi",
    "what is 2+2",
    "how many vacation days can I carry over?",
    "compare the 2022 and 2023 carryover rules",
    "what happens to unused days at the end of the year?",
    "who won the 2024 US election?",
]

for question in TEST_PROMPTS:
    out = run_agent(question)
    print("=" * 78)
    print("Q:", question)
    print(f"  source: {out['source']}   strategy: {out['strategy'] or '-'}")
    print(f"  queries:   {out['search_queries']}")
    print(f"  grade:     {out['grade'] or '-'}   attempts: {out['attempts']}")
    print(f"  answer:    {out['answer'][:220]}")
    print("  trace:")
    for step in out["trace"]:
        print("    -", step)

Q: hi
  source: direct   strategy: -
  queries:   []
  grade:     -   attempts: 0
  answer:    Hello!
  trace:
    - decide_retrieval -> direct
    - generate_answer -> done
Q: what is 2+2
  source: direct   strategy: -
  queries:   []
  grade:     -   attempts: 0
  answer:    2 + 2 equals 4.
  trace:
    - decide_retrieval -> direct
    - generate_answer -> done
Q: how many vacation days can I carry over?
  source: kb   strategy: rewrite
  queries:   ['vacation days carry over limit']
  grade:     good   attempts: 1
  answer:    You may carry over **up to 5 unused vacation (annual‑leave) days** into the next calendar year. The limit was reduced to five days starting with the 2023 policy year and any carried‑over days must be used by March 31 of 
  trace:
    - decide_retrieval -> kb
    - choose_strategy -> rewrite
    - rewrite_query -> 'vacation days carry over limit' (4 hits)
    - grade_chunks -> good
    - generate_answer -> done
Q: compare the 2022 and 2023 carryover rules
  sou